# Intrinsic semiconductor statistics from a 1D cosine potential

**Status:** Computational laboratory using the maintained PhysKit cosine-potential Bloch solver. The 1D model is idealized and is **not** a silicon band-structure calculation.

**Prerequisites:**

- [One-dimensional cosine-potential Bloch bands](../../../../docs/lecturenotes/solidstate/bloch1d/cosinepotential/index.md)
- [Maintained cosine-potential band laboratory](cosine-potential-bands.ipynb)

The notebook first calculates a self-consistent **one-dimensional** intrinsic carrier problem directly from the computed bands. It then evaluates the familiar **three-dimensional bulk, nondegenerate effective-density-of-states formulas** used in introductory device physics. These two dimensional models are deliberately reported separately; a 1D carrier density must not be presented as a 3D concentration.

Reference for the bulk intrinsic-semiconductor relations: S. M. Sze and K. K. Ng, *Physics of Semiconductor Devices*, 3rd ed. (Wiley, 2007). The exposition and code here are original and contain no reproduced textbook excerpts.


## Learning objectives

After running the notebook, you should be able to:

1. solve $V(x)=V_0\cos(2\pi x/a)$ with PhysKit over the first Brillouin zone;
2. identify valence- and conduction-band edges, the direct gap, and curvature effective masses;
3. construct a normalized 1D density of states;
4. solve intrinsic charge neutrality, $n_{1D}=p_{1D}$, using Fermi--Dirac occupations;
5. calculate the standard 3D bulk quantities $N_C$, $N_V$, $E_i$, $n_i$, conductivity, and resistivity under stated approximations; and
6. recognize which results belong to the idealized 1D model and which belong to a bulk semiconductor approximation.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.constants import Boltzmann, Planck, electron_mass, electron_volt, elementary_charge
from scipy.optimize import brentq
from scipy.special import expit

from projectkoios.physkit.solidstate.bloch1d.cell import BlochPrimitiveCell1D
from projectkoios.physkit.solidstate.bloch1d.cosinepotential.model import CosinePotentialBloch1D
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, UnitSystem, VectorQuantity

K_B_EV = Boltzmann / electron_volt
HBAR2_OVER_2ME_EV_A2 = 3.809_982_116_154_86
SPIN_DEGENERACY = 2.0


## 1. PhysKit cosine-potential model

We use one occupied valence band and one empty conduction band. With spin degeneracy two, a completely filled 1D band contains two electrons per primitive cell. The defaults $a=5.43$ Å and $V_0=1.12$ eV provide a silicon-scale length and gap for illustration only; fitting one cosine Fourier component does not reproduce silicon's crystal symmetry, valley degeneracy, indirect gap, or full density of states.


In [ ]:
# Editable model and numerical parameters.
lattice_constant_angstrom = 5.43
potential_amplitude_ev = 1.12
real_space_point_count = 64
k_point_count = 321  # odd: both equivalent zone edges and k=0 are included
band_count = 4

cell = BlochPrimitiveCell1D(
    oriented_length=ScalarQuantity(
        lattice_constant_angstrom,
        UnitSystem.METAL.length_unit,
    ),
    unit_system=UnitSystem.METAL,
)
model = CosinePotentialBloch1D(
    cell=cell,
    amplitude=ScalarQuantity(
        potential_amplitude_ev,
        UnitSystem.METAL.energy_unit,
    ),
)
reciprocal_unit = PhysicalUnit("1 / angstrom")
zone_edge = np.pi / lattice_constant_angstrom
k_plot = np.linspace(-zone_edge, zone_edge, k_point_count)

band_result = model.solve_bands(
    bloch_wave_numbers=VectorQuantity(k_plot, reciprocal_unit),
    point_count=real_space_point_count,
    band_count=band_count,
)
bands_ev = band_result.energies.magnitude

# The first and last samples are the same physical zone boundary.
assert np.allclose(bands_ev[0], bands_ev[-1], rtol=2e-8, atol=2e-8)


In [ ]:
valence_ev = bands_ev[:, 0]
conduction_ev = bands_ev[:, 1]
valence_edge_ev = float(np.max(valence_ev))
conduction_edge_ev = float(np.min(conduction_ev))
band_gap_ev = conduction_edge_ev - valence_edge_ev

# Fit E = E_edge + C (delta k)^2 on both sides of the equivalent zone edge.
fit_count = 10
delta_k = np.concatenate(
    (k_plot[:fit_count] + zone_edge, k_plot[-fit_count:] - zone_edge)
)
valence_fit_ev = np.concatenate((valence_ev[:fit_count], valence_ev[-fit_count:]))
conduction_fit_ev = np.concatenate(
    (conduction_ev[:fit_count], conduction_ev[-fit_count:])
)
valence_curvature_coefficient = np.polyfit(delta_k**2, valence_fit_ev, 1)[0]
conduction_curvature_coefficient = np.polyfit(
    delta_k**2, conduction_fit_ev, 1
)[0]
electron_effective_mass_ratio_1d = (
    HBAR2_OVER_2ME_EV_A2 / conduction_curvature_coefficient
)
hole_effective_mass_ratio_1d = (
    HBAR2_OVER_2ME_EV_A2 / -valence_curvature_coefficient
)

assert band_gap_ev > 0.0
assert electron_effective_mass_ratio_1d > 0.0
assert hole_effective_mass_ratio_1d > 0.0

pd.Series(
    {
        "valence edge Ev (eV)": valence_edge_ev,
        "conduction edge Ec (eV)": conduction_edge_ev,
        "direct gap Eg (eV)": band_gap_ev,
        "1D electron curvature mass / m0": electron_effective_mass_ratio_1d,
        "1D hole curvature mass / m0": hole_effective_mass_ratio_1d,
    },
    name="calculated",
)


In [ ]:
x_angstrom = np.linspace(
    0.0, lattice_constant_angstrom, 500, endpoint=False
)
potential_ev = potential_amplitude_ev * np.cos(
    2.0 * np.pi * x_angstrom / lattice_constant_angstrom
)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2))
axes[0].plot(x_angstrom, potential_ev)
axes[0].set(
    xlabel=r"$x$ (Å)",
    ylabel=r"$V(x)$ (eV)",
    title="Prescribed periodic potential",
)
for band_index in range(band_count):
    axes[1].plot(
        k_plot / zone_edge,
        bands_ev[:, band_index],
        label=f"band {band_index + 1}",
    )
axes[1].axhspan(valence_edge_ev, conduction_edge_ev, color="0.9", label="gap")
axes[1].set(
    xlabel=r"$k/(\pi/a)$",
    ylabel="Energy (eV)",
    title="PhysKit finite-difference Bloch bands",
)
axes[1].legend(ncol=2, fontsize=8)
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout()


## 2. One-dimensional density of states

For one spin-degenerate band,

$$
g_{1D}(E)=\frac{g_s}{2\pi}\int_{\mathrm{BZ}}\delta(E-E_n(k))\,dk,
$$

with units of states per energy per length. A narrow Gaussian represents the delta function numerically. Its integral over one complete band must approach $g_s/a=2/a$ states per unit length. The broadening changes the appearance of the van Hove peaks but not their intended normalization.


In [ ]:
# Drop the duplicate +pi/a endpoint for Brillouin-zone integration.
k_bz = k_plot[:-1]
bands_bz_ev = bands_ev[:-1]
delta_k_bz = 2.0 * zone_edge / k_bz.size
broadening_ev = 0.025
energy_grid_ev = np.linspace(
    float(np.min(bands_bz_ev[:, :2])) - 0.15,
    float(np.max(bands_bz_ev[:, :2])) + 0.15,
    1400,
)

def gaussian_dos_1d(band_energies_ev):
    differences = energy_grid_ev[:, None] - band_energies_ev[None, :]
    kernels = np.exp(-0.5 * (differences / broadening_ev) ** 2)
    kernels /= np.sqrt(2.0 * np.pi) * broadening_ev
    return SPIN_DEGENERACY * delta_k_bz / (2.0 * np.pi) * np.sum(
        kernels, axis=1
    )

valence_dos_per_ev_angstrom = gaussian_dos_1d(bands_bz_ev[:, 0])
conduction_dos_per_ev_angstrom = gaussian_dos_1d(bands_bz_ev[:, 1])
expected_states_per_angstrom = SPIN_DEGENERACY / lattice_constant_angstrom
valence_states_per_angstrom = np.trapezoid(
    valence_dos_per_ev_angstrom, energy_grid_ev
)
conduction_states_per_angstrom = np.trapezoid(
    conduction_dos_per_ev_angstrom, energy_grid_ev
)

assert np.isclose(
    valence_states_per_angstrom,
    expected_states_per_angstrom,
    rtol=2e-3,
)
assert np.isclose(
    conduction_states_per_angstrom,
    expected_states_per_angstrom,
    rtol=2e-3,
)

pd.Series(
    {
        "expected states per band (1/Å)": expected_states_per_angstrom,
        "integrated valence DOS (1/Å)": valence_states_per_angstrom,
        "integrated conduction DOS (1/Å)": conduction_states_per_angstrom,
    }
)


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(
    energy_grid_ev,
    valence_dos_per_ev_angstrom,
    label="valence band",
)
ax.plot(
    energy_grid_ev,
    conduction_dos_per_ev_angstrom,
    label="conduction band",
)
ax.axvspan(valence_edge_ev, conduction_edge_ev, color="0.9", label="gap")
ax.set(
    xlabel="Energy (eV)",
    ylabel=r"$g_{1D}(E)$ (states eV$^{-1}$ Å$^{-1}$)",
    title=f"Gaussian-broadened 1D DOS (sigma = {broadening_ev:.3f} eV)",
)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## 3. Exact intrinsic neutrality for the represented 1D bands

For the selected valence and conduction bands,

$$
n_{1D}(\mu,T)=\frac{g_s}{2\pi}\int_{\mathrm{BZ}}f(E_c(k),\mu,T)\,dk,
$$

$$
p_{1D}(\mu,T)=\frac{g_s}{2\pi}\int_{\mathrm{BZ}}[1-f(E_v(k),\mu,T)]\,dk.
$$

The intrinsic chemical potential is the root of $n_{1D}-p_{1D}=0$. This calculation uses the Fermi--Dirac function directly and does not assume parabolic bands or Boltzmann statistics. The result is a **linear density** in cm$^{-1}$.


In [ ]:
def fermi_occupation(energy_ev, chemical_potential_ev, temperature_k):
    return expit(
        (chemical_potential_ev - energy_ev) / (K_B_EV * temperature_k)
    )


def carrier_densities_1d(chemical_potential_ev, temperature_k):
    valence_occupation = fermi_occupation(
        bands_bz_ev[:, 0], chemical_potential_ev, temperature_k
    )
    conduction_occupation = fermi_occupation(
        bands_bz_ev[:, 1], chemical_potential_ev, temperature_k
    )
    state_weight = SPIN_DEGENERACY * delta_k_bz / (2.0 * np.pi)
    electrons_per_angstrom = state_weight * np.sum(conduction_occupation)
    holes_per_angstrom = state_weight * np.sum(1.0 - valence_occupation)
    return float(electrons_per_angstrom), float(holes_per_angstrom)


def intrinsic_state_1d(temperature_k):
    thermal_energy_ev = K_B_EV * temperature_k
    lower_bound = valence_edge_ev - 20.0 * thermal_energy_ev
    upper_bound = conduction_edge_ev + 20.0 * thermal_energy_ev

    def charge_imbalance(chemical_potential_ev):
        electrons, holes = carrier_densities_1d(
            chemical_potential_ev, temperature_k
        )
        return electrons - holes

    chemical_potential_ev = brentq(
        charge_imbalance,
        lower_bound,
        upper_bound,
        xtol=1e-13,
        rtol=1e-13,
    )
    electrons, holes = carrier_densities_1d(
        chemical_potential_ev, temperature_k
    )
    return {
        "temperature_K": float(temperature_k),
        "mu_i_eV": chemical_potential_ev,
        "n_1D_per_cm": electrons * 1.0e8,
        "p_1D_per_cm": holes * 1.0e8,
    }


In [ ]:
temperature_k = 300.0
intrinsic_1d = intrinsic_state_1d(temperature_k)
relative_neutrality_error = abs(
    intrinsic_1d["n_1D_per_cm"] - intrinsic_1d["p_1D_per_cm"]
) / max(intrinsic_1d["n_1D_per_cm"], intrinsic_1d["p_1D_per_cm"])
assert relative_neutrality_error < 2e-8

pd.Series(
    {
        **intrinsic_1d,
        "relative neutrality error": relative_neutrality_error,
    },
    name="exact 1D intrinsic state",
)


In [ ]:
mu_i_1d_ev = intrinsic_1d["mu_i_eV"]
fermi_on_energy_grid = fermi_occupation(
    energy_grid_ev, mu_i_1d_ev, temperature_k
)

fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.plot(
    energy_grid_ev,
    conduction_dos_per_ev_angstrom * fermi_on_energy_grid,
    label="occupied conduction DOS",
)
ax.plot(
    energy_grid_ev,
    valence_dos_per_ev_angstrom * (1.0 - fermi_on_energy_grid),
    label="empty valence DOS (holes)",
)
ax.axvline(mu_i_1d_ev, color="black", linestyle="--", label=r"$\mu_i$")
ax.set(
    xlabel="Energy (eV)",
    ylabel=r"thermally active DOS (states eV$^{-1}$ Å$^{-1}$)",
    title=f"Intrinsic excitations at {temperature_k:.0f} K",
)
ax.set_yscale("log")
ax.set_ylim(bottom=1e-14)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


In [ ]:
temperatures_1d_k = np.linspace(150.0, 600.0, 46)
intrinsic_1d_table = pd.DataFrame(
    [intrinsic_state_1d(value) for value in temperatures_1d_k]
)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].semilogy(
    intrinsic_1d_table["temperature_K"],
    intrinsic_1d_table["n_1D_per_cm"],
)
axes[0].set(
    xlabel="Temperature (K)",
    ylabel=r"$n_i^{1D}$ (cm$^{-1}$)",
    title="Exact 1D intrinsic carrier density",
)
axes[1].plot(
    intrinsic_1d_table["temperature_K"],
    intrinsic_1d_table["mu_i_eV"],
    label=r"$\mu_i$",
)
axes[1].axhline(valence_edge_ev, color="C2", linestyle="--", label=r"$E_v$")
axes[1].axhline(conduction_edge_ev, color="C3", linestyle="--", label=r"$E_c$")
axes[1].set(
    xlabel="Temperature (K)",
    ylabel="Energy (eV)",
    title="Intrinsic chemical potential",
)
for axis in axes:
    axis.grid(alpha=0.25)
axes[1].legend()
fig.tight_layout()


## 4. Bulk intrinsic-semiconductor relations (3D, nondegenerate)

The standard parabolic-band, Maxwell--Boltzmann approximation gives

$$
N_C=2\left(\frac{2\pi m_n^* k_BT}{h^2}\right)^{3/2},\qquad
N_V=2\left(\frac{2\pi m_p^* k_BT}{h^2}\right)^{3/2},
$$

$$
n_i=\sqrt{N_CN_V}\exp\left(-\frac{E_g}{2k_BT}\right),
$$

and

$$
E_i=\frac{E_C+E_V}{2}+\frac{3}{4}k_BT\ln\left(\frac{m_p^*}{m_n^*}\right).
$$

For an intrinsic semiconductor, $n=p=n_i$, $np=n_i^2$, and, when drift mobilities are supplied,

$$
\sigma_i=q n_i(\mu_n+\mu_p),\qquad \rho_i=\sigma_i^{-1}.
$$

The following silicon-like defaults are independent bulk parameters. They are **not** inferred from the 1D DOS. The mobility values are illustrative room-temperature inputs; no temperature-dependent scattering model is asserted.


In [ ]:
# Editable bulk silicon-like parameters.
gap_zero_k_ev = 1.17
varshni_alpha_ev_per_k = 4.73e-4
varshni_beta_k = 636.0
electron_dos_mass_ratio = 1.08
hole_dos_mass_ratio = 0.56
electron_mobility_cm2_per_vs = 1350.0
hole_mobility_cm2_per_vs = 480.0


def silicon_like_gap_ev(temperature_k):
    return gap_zero_k_ev - (
        varshni_alpha_ev_per_k
        * temperature_k**2
        / (temperature_k + varshni_beta_k)
    )


def bulk_intrinsic_state(temperature_k):
    gap_ev = silicon_like_gap_ev(temperature_k)
    electron_mass_kg = electron_dos_mass_ratio * electron_mass
    hole_mass_kg = hole_dos_mass_ratio * electron_mass
    nc_per_m3 = 2.0 * (
        2.0 * np.pi * electron_mass_kg * Boltzmann * temperature_k / Planck**2
    ) ** 1.5
    nv_per_m3 = 2.0 * (
        2.0 * np.pi * hole_mass_kg * Boltzmann * temperature_k / Planck**2
    ) ** 1.5
    nc_per_cm3 = nc_per_m3 / 1.0e6
    nv_per_cm3 = nv_per_m3 / 1.0e6
    intrinsic_level_above_ev_ev = (
        0.5 * gap_ev
        + 0.75
        * K_B_EV
        * temperature_k
        * np.log(hole_dos_mass_ratio / electron_dos_mass_ratio)
    )
    intrinsic_density_per_cm3 = np.sqrt(nc_per_cm3 * nv_per_cm3) * np.exp(
        -gap_ev / (2.0 * K_B_EV * temperature_k)
    )
    conductivity_per_ohm_cm = (
        elementary_charge
        * intrinsic_density_per_cm3
        * (electron_mobility_cm2_per_vs + hole_mobility_cm2_per_vs)
    )
    resistivity_ohm_cm = 1.0 / conductivity_per_ohm_cm
    electron_thermal_speed_m_per_s = np.sqrt(
        3.0 * Boltzmann * temperature_k / electron_mass_kg
    )
    hole_thermal_speed_m_per_s = np.sqrt(
        3.0 * Boltzmann * temperature_k / hole_mass_kg
    )
    return {
        "temperature_K": float(temperature_k),
        "Eg_eV": gap_ev,
        "Nc_per_cm3": nc_per_cm3,
        "Nv_per_cm3": nv_per_cm3,
        "Ei_minus_Ev_eV": intrinsic_level_above_ev_ev,
        "ni_per_cm3": intrinsic_density_per_cm3,
        "conductivity_per_ohm_cm": conductivity_per_ohm_cm,
        "resistivity_ohm_cm": resistivity_ohm_cm,
        "electron_thermal_speed_m_per_s": electron_thermal_speed_m_per_s,
        "hole_thermal_speed_m_per_s": hole_thermal_speed_m_per_s,
    }


def bulk_boltzmann_carriers(fermi_level_above_ev_ev, temperature_k):
    state = bulk_intrinsic_state(temperature_k)
    electron_density = state["Nc_per_cm3"] * np.exp(
        -(state["Eg_eV"] - fermi_level_above_ev_ev)
        / (K_B_EV * temperature_k)
    )
    hole_density = state["Nv_per_cm3"] * np.exp(
        -fermi_level_above_ev_ev / (K_B_EV * temperature_k)
    )
    return electron_density, hole_density


In [ ]:
bulk_300k = bulk_intrinsic_state(300.0)
bulk_n_300k, bulk_p_300k = bulk_boltzmann_carriers(
    bulk_300k["Ei_minus_Ev_eV"], 300.0
)

assert np.isclose(bulk_n_300k, bulk_300k["ni_per_cm3"], rtol=2e-13)
assert np.isclose(bulk_p_300k, bulk_300k["ni_per_cm3"], rtol=2e-13)
assert np.isclose(
    bulk_n_300k * bulk_p_300k,
    bulk_300k["ni_per_cm3"] ** 2,
    rtol=3e-13,
)

pd.Series(
    {
        **bulk_300k,
        "n_at_Ei_per_cm3": bulk_n_300k,
        "p_at_Ei_per_cm3": bulk_p_300k,
        "np_over_ni_squared": (
            bulk_n_300k * bulk_p_300k / bulk_300k["ni_per_cm3"] ** 2
        ),
    },
    name="3D nondegenerate bulk approximation at 300 K",
)


In [ ]:
bulk_temperatures_k = np.linspace(150.0, 600.0, 91)
bulk_table = pd.DataFrame(
    [bulk_intrinsic_state(value) for value in bulk_temperatures_k]
)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))
axes[0].plot(bulk_table["temperature_K"], bulk_table["Eg_eV"])
axes[0].set(xlabel="Temperature (K)", ylabel=r"$E_g$ (eV)", title="Gap input")
axes[1].semilogy(
    bulk_table["temperature_K"],
    bulk_table["Nc_per_cm3"],
    label=r"$N_C$",
)
axes[1].semilogy(
    bulk_table["temperature_K"],
    bulk_table["Nv_per_cm3"],
    label=r"$N_V$",
)
axes[1].semilogy(
    bulk_table["temperature_K"],
    bulk_table["ni_per_cm3"],
    label=r"$n_i$",
)
axes[1].set(
    xlabel="Temperature (K)",
    ylabel=r"Concentration (cm$^{-3}$)",
    title="Bulk density scales",
)
axes[1].legend()
axes[2].plot(
    bulk_table["temperature_K"],
    bulk_table["Ei_minus_Ev_eV"],
    label=r"$E_i-E_V$",
)
axes[2].plot(
    bulk_table["temperature_K"],
    0.5 * bulk_table["Eg_eV"],
    linestyle="--",
    label="midgap",
)
axes[2].set(
    xlabel="Temperature (K)",
    ylabel="Energy (eV)",
    title="Intrinsic level",
)
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout()


## 5. Interpretation and limitations

- The cosine potential produces a direct first gap at the 1D zone boundary. It demonstrates gap opening and band-edge curvature, but it is not a material-specific representation of silicon.
- The exact 1D calculation uses the computed nonparabolic bands and returns carriers per unit **length**.
- The 3D calculation uses independent bulk effective-density-of-states masses and returns carriers per unit **volume**. Its Maxwell--Boltzmann form is appropriate only when the Fermi level is several $k_BT$ from both band edges.
- The conductivity estimate assumes supplied drift mobilities. It omits a microscopic scattering calculation and should not be used as a temperature-dependent mobility model.
- No donor/acceptor ionization, band-gap narrowing, degeneracy correction, recombination-generation kinetics, surfaces, contacts, or spatial electrostatics are included. Those are distinct physical models, not missing algebra in the intrinsic equilibrium calculation.

### Exercises

1. Vary $V_0$ and verify the weak-potential relation between the first gap and $|V_0|$.
2. Increase the real-space point count and quantify convergence of the gap and curvature masses.
3. Compare the exact 1D intrinsic chemical potential with a 1D parabolic-band Boltzmann approximation.
4. Change $m_n^*/m_0$ and $m_p^*/m_0$ in the bulk model and explain the shift of $E_i$ away from midgap.
5. Replace the constant mobilities only after selecting and documenting a temperature-dependent scattering model.
